# Combining DataFrames

## Introduction

Combining data from multiple sources is a foundational data cleaning task. This notebook covers two approaches: **concatenation** (stacking DataFrames end-to-end) and **joins** (merging on a shared key).

## Objectives

You will be able to:

- Use `pd.concat()` to combine DataFrames with and without join conditions
- Understand primary keys and set a DataFrame index
- Distinguish between inner, outer, left, and right joins
- Apply `.join()` to merge DataFrames in pandas

In [ ]:
import pandas as pd

---

## Concatenating DataFrames

Concatenation appends one DataFrame onto the end of another — like extending a list. Pass a list of DataFrames to `pd.concat()`:

```python
big_df = pd.concat([df1, df2, df3])
```

![concat diagram](assets/combining_dataframes_pandas/Image_197_concat.png)

In [ ]:
df1 = pd.DataFrame({'A': ['A0', 'A1', 'A2', 'A3'],
                    'B': ['B0', 'B1', 'B2', 'B3'],
                    'C': ['C0', 'C1', 'C2', 'C3'],
                    'D': ['D0', 'D1', 'D2', 'D3']},
                    index=[0, 1, 2, 3])

df2 = pd.DataFrame({'A': ['A4', 'A5', 'A6', 'A7'],
                    'B': ['B4', 'B5', 'B6', 'B7'],
                    'C': ['C4', 'C5', 'C6', 'C7'],
                    'D': ['D4', 'D5', 'D6', 'D7']},
                    index=[4, 5, 6, 7])

df3 = pd.DataFrame({'A': ['A8', 'A9', 'A10', 'A11'],
                    'B': ['B8', 'B9', 'B10', 'B11'],
                    'C': ['C8', 'C9', 'C10', 'C11'],
                    'D': ['D8', 'D9', 'D10', 'D11']},
                    index=[8, 9, 10, 11])

df1

Concatenate `df1`, `df2`, and `df3` into a single DataFrame.

In [ ]:
combined_df = pd.concat([df1, df2, df3])
combined_df

### Join conditions in concatenation

By default `pd.concat()` performs an **outer** join on columns — keeping all columns from all DataFrames and filling gaps with `NaN`. Pass `join='inner'` to keep only the columns shared by all DataFrames.

You can also concatenate **side-by-side** with `axis=1`. The example below creates `df4` with overlapping index values but different columns, then inner-joins it onto `df1` — only the two rows whose indexes appear in both DataFrames survive.

In [ ]:
df4 = pd.DataFrame({'B': ['B2', 'B3', 'B6', 'B7'],
                    'D': ['D2', 'D3', 'D6', 'D7'],
                    'F': ['F2', 'F3', 'F6', 'F7']},
                    index=[2, 3, 6, 7])

# Only rows 2 and 3 have matching indexes in both df1 and df4
pd.concat([df1, df4], axis=1, join='inner')

---

## Keys and Indexes

Every table has a **primary key** — a column that uniquely identifies each row. In pandas, the index plays this role. When joining DataFrames, rows are aligned by matching index values (or a shared column used as a **foreign key**).

To set a column as the index without mutation:

```python
df = pd.read_csv('file.csv').set_index('id_column')
```

---

## Types of Joins

Joins combine two DataFrames **horizontally**, aligning rows by a shared key. The four types can be visualised as Venn diagrams:

![join types](assets/combining_dataframes_pandas/Image_198_joins.png)

| Join | Returns |
|------|---------|
| **Inner** | Only rows with matching keys in **both** tables |
| **Outer** | All rows from **both** tables; `NaN` where there is no match |
| **Left** | All rows from the left table + matching rows from the right |
| **Right** | All rows from the right table + matching rows from the left |

In pandas, call `.join()` on the left DataFrame:

```python
joined_df = df1.join(df2, how='inner')  # default is 'left'
```

If both tables share column names, pass `lsuffix=` or `rsuffix=` to avoid a naming collision.

---

## Practice: Hearthstone Cards Dataset

The [Hearthstone cards dataset](https://www.kaggle.com/jeradrose/hearthstone-cards) splits card data across five tables linked by `card_id`. Load all five and set `card_id` as the index on each.

In [ ]:
cards_df             = pd.read_csv('data/combining_dataframes_pandas_lab/cards.csv').set_index('card_id')
dust_df              = pd.read_csv('data/combining_dataframes_pandas_lab/dust.csv').set_index('card_id')
entourages_df        = pd.read_csv('data/combining_dataframes_pandas_lab/entourages.csv').set_index('card_id')
mechanics_df         = pd.read_csv('data/combining_dataframes_pandas_lab/mechanics.csv').set_index('card_id')
play_requirements_df = pd.read_csv('data/combining_dataframes_pandas_lab/play_requirements.csv').set_index('card_id')

print(f"cards_df: {len(cards_df)} rows")
cards_df.head()

### Inner join

Join `cards_df` with `mechanics_df` using an inner join. The result contains only cards that appear in both tables — notice how many rows are filtered out.

In [ ]:
cards_with_mechanics_df = cards_df.join(mechanics_df, how='inner')
print(f"cards_df: {len(cards_df)} rows  →  inner join: {len(cards_with_mechanics_df)} rows")
cards_with_mechanics_df.head()

### Left join

Perform a left join of `cards_with_mechanics_df` and `play_requirements_df`. Every row from the left table is preserved; rows with no matching play requirement get `NaN` in the new columns.

In [ ]:
left_join_df = cards_with_mechanics_df.join(play_requirements_df)  # default how='left'
print(f"inner join: {len(cards_with_mechanics_df)} rows  →  left join: {len(left_join_df)} rows")
left_join_df.head()

### Outer join

Perform an outer join of `cards_df` and `dust_df`. Both tables contain a `cost` column — pass `rsuffix='_dust'` to resolve the naming collision. The result includes every card regardless of whether it has a dust cost entry.

In [ ]:
outer_join_df = cards_df.join(dust_df, rsuffix='_dust', how='outer')
print(f"cards_df: {len(cards_df)} rows  →  outer join: {len(outer_join_df)} rows")
outer_join_df.head()

---

## Summary

In this notebook you learned how to:

- Concatenate DataFrames with `pd.concat()`, controlling direction (`axis`) and column overlap (`join`)
- Set a DataFrame index to enable key-based joining
- Apply all four join types — inner, outer, left, and right — using `.join()`

Next: [02 — Lambda Functions](02_lambda_functions.ipynb)